# Linear Regression with Multiple Outputs — worked study copy

Answers prepared for revision with AI assistance. IBM / Skills Network source notices and embedded instructor explanations are retained. This is not a course submission.

Code completion and execution are tracked separately in the [solution progress report](../../../docs/SOLUTION_PROGRESS.md). Training may require a separate environment, external data, compute or provider credentials.

[Original reference](../../../courses/04-pytorch-foundations/notebooks/linear-regression-with-multiple-outputs.ipynb) · [Course notes](../../../courses/04-pytorch-foundations/README.md)

**BOUNDED SMOKE RUN:** seed 42; CSVs sampled to at most 5,000 rows; one grid setting with two folds; training budgets capped; DataLoaders limited to 12 batches. These outputs are execution checks, not full-run model evaluations.

In [1]:
import os, itertools, random
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['TF_NUM_INTEROP_THREADS'] = '1'
os.environ['TF_NUM_INTRAOP_THREADS'] = '1'
import numpy as np
np.random.seed(42)
random.seed(42)
import pandas as pd
_read_csv = pd.read_csv
def _bounded_csv(*args, **kwargs):
    data = _read_csv(*args, **kwargs)
    return data.sample(n=5000,random_state=42) if len(data)>5000 else data
pd.read_csv = _bounded_csv
from sklearn.model_selection import GridSearchCV
_grid_fit = GridSearchCV.fit
def _bounded_grid(self, *args, **kwargs):
    grids = self.param_grid if isinstance(self.param_grid,list) else [self.param_grid]
    self.param_grid = [{k:list(v)[:1] for k,v in grid.items()} for grid in grids[:1]]
    self.cv = 2
    self.n_jobs = 1
    self.verbose = 0
    return _grid_fit(self,*args,**kwargs)
GridSearchCV.fit = _bounded_grid
try:
    import torch
except ImportError:
    torch = None
if torch is not None:
    torch.manual_seed(42)
    torch.set_num_threads(1)
    _loader_iter = torch.utils.data.DataLoader.__iter__
    def _bounded_loader(self):
        return itertools.islice(_loader_iter(self),12)
    torch.utils.data.DataLoader.__iter__ = _bounded_loader


<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# Lab: Linear Regression with Multiple Outputs


<h2>Objective</h2><ul><li> How to make a prediction using multiple samples.</li></ul> 



## Table of Contents
In this lab, we will  review how to make a prediction for Linear Regression with Multiple Outputs. 

<div class="alert alert-block alert-info" style="margin-top: 20px">
<ul>
<li><a href="#ref1">Class Linear </a></li>
    </ul>
<p></p>
Estimated Time Needed: <strong>15 min</strong>
</div>

<hr>


<a id="ref1"></a>
<a name="ref1"><h2 align=center>Class Linear  </h2></a>


In [2]:
%%time
# Install this lab's dependencies in its isolated environment before execution.

CPU times: user 1 µs, sys: 0 ns, total: 1 µs
Wall time: 1.91 µs


In [3]:
from torch import nn
import torch

Set the random seed:


In [4]:
torch.manual_seed(1)

Set the random seed:


In [5]:
class linear_regression(nn.Module):
    def __init__(self,input_size,output_size):
        super(linear_regression,self).__init__()
        self.linear=nn.Linear(input_size,output_size)
    def forward(self,x):
        yhat=self.linear(x)
        return yhat

create a linear regression  object, as our input and output will be two we set the parameters accordingly 


In [6]:
model=linear_regression(1,10)
model(torch.tensor([1.0]))

tensor([ 0.7926, -0.3920,  0.1714,  0.0797, -1.0143,  0.5097, -0.0608,  0.5047,
         1.0132,  0.1887], grad_fn=<ViewBackward0>)

we can use the diagram to represent the model or object 


<img src="https://ibm.box.com/shared/static/icmwnxru7nytlhnq5x486rffea9ncpk7.png" width="600," align="center">


we can see the parameters 


In [7]:
list(model.parameters())

[Parameter containing:
 tensor([[ 0.5153],
         [-0.4414],
         [-0.1939],
         [ 0.4694],
         [-0.9414],
         [ 0.5997],
         [-0.2057],
         [ 0.5087],
         [ 0.1390],
         [-0.1224]], requires_grad=True),
 Parameter containing:
 tensor([ 0.2774,  0.0493,  0.3652, -0.3897, -0.0729, -0.0900,  0.1449, -0.0040,
          0.8742,  0.3112], requires_grad=True)]

we can create a tensor with two rows representing one sample of data


In [8]:
x=torch.tensor([[1.0]])

we can make a prediction 


In [9]:
yhat=model(x)
yhat

tensor([[ 0.7926, -0.3920,  0.1714,  0.0797, -1.0143,  0.5097, -0.0608,  0.5047,
          1.0132,  0.1887]], grad_fn=<AddmmBackward0>)

each row in the following tensor represents a different sample 


In [10]:
X=torch.tensor([[1.0],[1.0],[3.0]])

we can make a prediction using multiple samples 


In [11]:
Yhat=model(X)
Yhat

tensor([[ 0.7926, -0.3920,  0.1714,  0.0797, -1.0143,  0.5097, -0.0608,  0.5047,
          1.0132,  0.1887],
        [ 0.7926, -0.3920,  0.1714,  0.0797, -1.0143,  0.5097, -0.0608,  0.5047,
          1.0132,  0.1887],
        [ 1.8232, -1.2748, -0.2164,  1.0184, -2.8972,  1.7091, -0.4722,  1.5222,
          1.2912, -0.0561]], grad_fn=<AddmmBackward0>)

the following figure represents the operation, where the red and blue  represents the different parameters, and the different shades of green represent  different samples.


 <img src="https://ibm.box.com/shared/static/768cul6pj8hc93uh9ujpajihnp8xdukx.png" width="600," align="center">


# About the Authors:  

 [Joseph Santarcangelo]( https://www.linkedin.com/in/joseph-s-50398b136/) has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD.
 
Other contributors: [Michelle Carey](  https://www.linkedin.com/in/michelleccarey/) 


<!--
## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2020-09-23  | 2.0  | Shubham  |  Migrated Lab to Markdown and added to course repo in GitLab |

-->


## <h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>
